# Lab 7: Chain-of-Thought e Self-Consistency

## Lab 7: Chain-of-Thought e Self-Consistency — o mecanismo, não a mágica

**Nota honesta sobre o modelo:** usamos `sshleifer/tiny-gpt2` (~100k
parâmetros, pesos essencialmente aleatórios) — o mesmo modelo "de
mecânica" das Semanas 2-3. Ele **não vai raciocinar bem** (é pequeno
demais pra entender matemática). O objetivo aqui não é provar que CoT
melhora accuracy (isso é bem documentado na literatura, com modelos de
verdade) — é ver o **código de self-consistency rodando de ponta a
ponta**: múltiplas amostras reais, voto majoritário real. Trocar
`tiny-gpt2` por um modelo de produção não muda uma linha desse código, só
a qualidade das respostas.

In [1]:
!pip install -q transformers torch

import re
import torch
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()
print(f"✓ Modelo carregado: {MODEL_NAME}")

✓ Modelo carregado: sshleifer/tiny-gpt2


### 1. Dataset de problemas com resposta verificável

In [2]:
problems = [
    {"question": "2 + 2 =", "answer": 4},
    {"question": "10 - 3 =", "answer": 7},
    {"question": "5 * 2 =", "answer": 10},
]
print(f"✓ {len(problems)} problemas com resposta numérica verificável")

✓ 3 problemas com resposta numérica verificável


### 2. Geração + extração de número

In [3]:
def extract_number(text: str):
    numbers = re.findall(r"-?\d+\.?\d*", text)
    return float(numbers[0]) if numbers else None

def generate(prompt: str, temperature: float = 0.7, max_new_tokens: int = 12) -> str:
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=temperature, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

### 3. Zero-shot vs Chain-of-Thought (prompting)

In [4]:
correct_zero_shot, correct_cot = 0, 0
for p in problems:
    zs = extract_number(generate(f"{p['question']}", temperature=0.3))
    cot = extract_number(generate(f"Let's think step by step. {p['question']}", temperature=0.3))
    correct_zero_shot += int(zs == p["answer"])
    correct_cot += int(cot == p["answer"])
    print(f"'{p['question']}' → zero-shot={zs}, CoT={cot}, real={p['answer']}")

print(f"\nZero-shot: {correct_zero_shot}/{len(problems)} | CoT: {correct_cot}/{len(problems)}")
print("(accuracy baixa é esperada — tiny-gpt2 não entende matemática, é só a mecânica que importa aqui)")

'2 + 2 =' → zero-shot=236.0, CoT=None, real=4
'10 - 3 =' → zero-shot=236.0, CoT=None, real=7
'5 * 2 =' → zero-shot=None, CoT=4.0, real=10

Zero-shot: 0/3 | CoT: 0/3
(accuracy baixa é esperada — tiny-gpt2 não entende matemática, é só a mecânica que importa aqui)


**Resultado esperado:** accuracy provavelmente 0 ou quase 0 nos dois —
`tiny-gpt2` não tem capacidade real de aritmética. O que confirma que o
código *roda* e produz um número extraído (mesmo que errado).

### 4. Self-consistency — o mecanismo de verdade

In [5]:
def self_consistency(question: str, n: int = 5) -> float:
    answers = [extract_number(generate(question, temperature=1.0)) for _ in range(n)]
    answers = [a for a in answers if a is not None]
    print(f"    amostras: {answers}")
    if not answers:
        return None
    return Counter(answers).most_common(1)[0][0]

correct_sc = 0
for p in problems:
    print(f"'{p['question']}':")
    pred = self_consistency(p["question"])
    hit = pred == p["answer"]
    correct_sc += int(hit)
    print(f"  → voto majoritário: {pred} (real: {p['answer']})")

print(f"\nSelf-consistency: {correct_sc}/{len(problems)}")

'2 + 2 =':
    amostras: [236.0, 236.0]
  → voto majoritário: 236.0 (real: 4)
'10 - 3 =':
    amostras: [653.0, 653.0]
  → voto majoritário: 653.0 (real: 7)
'5 * 2 =':
    amostras: [448.0, 236.0, 653.0]
  → voto majoritário: 448.0 (real: 10)

Self-consistency: 0/3


**Resultado esperado:** pra cada problema, 5 amostras diferentes são
geradas (com `temperature=1.0`, então elas variam de verdade) e o
`Counter` escolhe a mais frequente — o mecanismo de "gerar várias vezes e
votar" está genuinamente funcionando, mesmo que o modelo pequeno não saiba
matemática o suficiente pra convergir na resposta certa.

**O que fica pra produção:** com um modelo de verdade (Fase 1 usa Claude
via API — troque `generate()` por uma chamada real à Anthropic API com
`temperature` variável), esse mesmo código de self-consistency melhora
accuracy de forma mensurável — é uma técnica bem estabelecida na
literatura (Wang et al., 2022, "Self-Consistency Improves Chain of Thought
Reasoning").

**Próximos passos:** Semana 8 aborda como *treinar* o modelo pra ter
comportamentos preferidos (não só induzir via prompt, como fizemos aqui).